<a href="https://colab.research.google.com/github/mab2004/BERT-News-Topic-Classifier/blob/main/BERT_News_Topic_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup and Installation

In [ ]:
!pip install transformers datasets evaluate accelerate

## Dataset Loading and Preprocessing

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

## Load the AG News dataset
dataset = load_dataset("ag_news")

## Load the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

## Define a function to tokenize the text
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

## Apply the tokenization to the entire dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

## Set the format for PyTorch and remove unnecessary columns
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

## Create smaller splits for faster training, if needed
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(10000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

## Model Development & Training


In [ ]:
## Load the pre-trained BERT model with a classification head
## There are 4 labels in the AG News dataset
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

## Define the training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none"
)

## Define the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
)

## Start training the model
trainer.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.397400,0.273024
2,0.221200,0.281415


Epoch,Training Loss,Validation Loss
1,0.397400,0.273024
2,0.221200,0.281415
3,0.162200,0.297107


TrainOutput(global_step=1875, training_loss=0.23032461547851563, metrics={'train_runtime': 2894.6905, 'train_samples_per_second': 10.364, 'train_steps_per_second': 0.648, 'total_flos': 7893473402880000.0, 'train_loss': 0.23032461547851563, 'epoch': 3.0})

## Evaluation with Relevant Metrics

In [ ]:
import evaluate

## Load the evaluation metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

## Perform predictions on the evaluation dataset
predictions = trainer.predict(small_eval_dataset)
logits = predictions.predictions
labels = predictions.label_ids

## Get the predicted labels by taking the argmax of the logits
predicted_labels = np.argmax(logits, axis=-1)

## Compute the accuracy and F1-score
accuracy = accuracy_metric.compute(predictions=predicted_labels, references=labels)
f1_score = f1_metric.compute(predictions=predicted_labels, references=labels, average="weighted")

print(f"Accuracy: {accuracy['accuracy']:.4f}")
print(f"F1-Score: {f1_score['f1']:.4f}")

## You can also use scikit-learn's classification_report for a detailed report
from sklearn.metrics import classification_report

print("\nDetailed Classification Report:")
print(classification_report(labels, predicted_labels, target_names=["World", "Sports", "Business", "Sci/Tech"]))

Accuracy: 0.9200
F1-Score: 0.9204

Detailed Classification Report:
              precision    recall  f1-score   support

       World       0.96      0.91      0.93       266
      Sports       0.98      0.98      0.98       246
    Business       0.90      0.88      0.89       246
    Sci/Tech       0.84      0.91      0.88       242

    accuracy                           0.92      1000
   macro avg       0.92      0.92      0.92      1000
weighted avg       0.92      0.92      0.92      1000



## Save the model and tokenizer

In [ ]:
trainer.save_model("./ag_news_bert_model")
tokenizer.save_pretrained("./ag_news_bert_model")

('./ag_news_bert_model/tokenizer_config.json',
 './ag_news_bert_model/special_tokens_map.json',
 './ag_news_bert_model/vocab.txt',
 './ag_news_bert_model/added_tokens.json',
 './ag_news_bert_model/tokenizer.json')

## Deployment Files Setup

In [25]:
%%writefile app.py
import streamlit as st
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# Define the directory where your model was saved
MODEL_DIR = "./ag_news_bert_model"

## Load the fine-tuned model and tokenizer
@st.cache_resource
def get_model():
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

    # Create the text classification pipeline
    return pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer
    )

classifier = get_model()

# Map the raw output labels (LABEL_0, LABEL_1, etc.) to the actual topic names
LABEL_MAP = {
    "LABEL_0": "World",
    "LABEL_1": "Sports",
    "LABEL_2": "Business",
    "LABEL_3": "Sci/Tech"
}

st.title("News Topic Classifier 📰")
st.write("Enter a news headline below to classify its topic using the fine-tuned BERT model.")

user_input = st.text_input("News Headline:")

if user_input:
    # Run the classification
    with st.spinner('Classifying...'):
        result = classifier(user_input)

    predicted_label_key = result[0]['label']
    predicted_label = LABEL_MAP.get(predicted_label_key, "Unknown")
    score = result[0]['score']

    st.subheader("Prediction:")
    st.markdown(f"**Topic:** <span style='font-size: 24px;'>{predicted_label}</span>", unsafe_allow_html=True)
    st.write(f"**Confidence Score:** {score:.4f}")

Overwriting app.py


In [ ]:
%%writefile requirements.txt
streamlit
transformers
torch
datasets
accelerate
scikit-learn

Writing requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%bash
# Create a new folder in your Drive (e.g., in 'My Drive/Colab Notebooks/NewsClassifier')
DRIVE_PATH="/content/drive/MyDrive/Colab Notebooks/NewsClassifier"
mkdir -p "$DRIVE_PATH"

# Copy the entire model folder and deployment files
cp -r ./ag_news_bert_model "$DRIVE_PATH/"
cp app.py "$DRIVE_PATH/"
cp requirements.txt "$DRIVE_PATH/"

echo "All files saved to $DRIVE_PATH"

All files saved to /content/drive/MyDrive/Colab Notebooks/NewsClassifier


## Running the Streamlit App

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%%bash
# The path where the files are stored
DRIVE_PATH="/content/drive/MyDrive/Colab Notebooks/NewsClassifier"
cp -r "$DRIVE_PATH/ag_news_bert_model" ./
cp "$DRIVE_PATH/app.py" ./
cp "$DRIVE_PATH/requirements.txt" ./

In [ ]:
!pip install -r requirements.txt

In [6]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 5s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [27]:
!curl https://loca.lt/mytunnelpassword

34.138.9.65

In [30]:
import time

# 1. KILL any existing Streamlit process forcefully 💀
print("Killing any existing Streamlit process...")
!pkill -f streamlit
time.sleep(2) # Wait a moment for the process to fully clear

# 2. START Streamlit in the background
print("Starting Streamlit server...")
# The '&' runs it in the background; stdout/stderr are piped to ignore
!nohup streamlit run app.py --server.port 8501 --server.enableCORS false > /dev/null 2>&1 &

# 3. CRUCIAL: Wait for the large BERT model to load on the CPU ⏳
# This is a critical step for large models on a CPU runtime.
print("Waiting 30 seconds for the BERT model to load...")
time.sleep(30)

print("Attempting to establish public tunnel (lt)...")

# 4. ESTABLISH the localtunnel connection and get the URL
!lt --port 8501

Killing any existing Streamlit process...
Starting Streamlit server...
Waiting 30 seconds for the BERT model to load...
Attempting to establish public tunnel (lt)...
your url is: https://spotty-peaches-lie.loca.lt
^C
